# Fork comparison

Compared on 2026-11-06 against `LLusvarghi/MoReV2X` at `f3b8b5c` (v4.1, plus a README commit of 2024-03-15). GitHub lists 8 forks. All of their branches were fetched into this clone as `fork-*` remotes, which have no push URL.

## Verdict

**Three forks hold code: kaefcatcher, vivatttt, carlosrisma. The other five hold nothing to merge.** No fork can be merged as a whole: each mixes useful changes with something that would damage results.

| Fork | Branch | Ahead / behind upstream | Content | Action |
|---|---|---|---|---|
| [kaefcatcher/MoReV2X_V2](https://github.com/kaefcatcher/MoReV2X_V2) | `main` | 24 / 0 | SUMO CSV traces, `Pkeep`, YAML sweeps, AoI metrics, 738 MB of data | Cherry-pick 3 items |
| | `develop` | 6 / 0 | Older subset of `main` | Ignore |
| [vivatttt/MoReV2X](https://github.com/vivatttt/MoReV2X) | `aoi-aware-congestion-control` | 6 / 0 | Adaptive RRI and Pkeep from sensed occupancy, AoI and CBR metrics | Cherry-pick 2 fixes, rewrite the algorithm if wanted |
| | `feature/aoi-aware-congestion-control-algorithm` | 1 / 0 | First draft of the same | Ignore |
| | `main` | 0 / 0 | Identical to upstream | Nothing |
| [carlosrisma/MoReV2X](https://github.com/carlosrisma/MoReV2X) | `main` | 2 / 1 | Module ported to ns-3.36 | Keep as a reference for a later migration |
| [maliksaad84/MoReV2X](https://github.com/maliksaad84/MoReV2X) | `main` | 0 / 1 | v4.1 before upstream's README commit | Nothing |
| [xxxxxy211/MoReV2X](https://github.com/xxxxxy211/MoReV2X) | `main` | 0 / 0 | Identical to upstream | Nothing |
| [NawazAli20/MoReV2X](https://github.com/NawazAli20/MoReV2X), [tfehren/MoReV2X](https://github.com/tfehren/MoReV2X) | `main` | 7 / 10 | 2021 snapshots. The 7 commits are upstream's own README edits, and they miss all 3,365 files added since | Nothing |
| [15738897318/MoReV2X](https://github.com/15738897318/MoReV2X) | `main` | 1 / 10 | Same, 1 README commit | Nothing |

Check the numbers yourself:

In [1]:
import os, subprocess, sys
sys.path.insert(0, os.path.abspath("../../tools"))
import morev2x_results as mr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 90)
mr.apply_chart_style()
os.chdir(mr.ROOT)

In [2]:
def git(*args):
    return subprocess.run(("git",) + args, capture_output=True, text=True, cwd=mr.ROOT).stdout.strip()

rows = []
for branch in git("for-each-ref", "--format=%(refname:short)", "refs/remotes").split():
    if not branch.startswith("fork-") or branch.count("/") == 0:
        continue
    base = git("merge-base", "upstream/main", branch)
    rows.append({"branch": branch, "ahead": int(git("rev-list", "--count", "upstream/main.." + branch)),
                 "behind": int(git("rev-list", "--count", branch + "..upstream/main")),
                 "diff": git("diff", "--shortstat", base, branch)})
pd.DataFrame(rows).set_index("branch")

,ahead,behind,diff
branch,,,
fork-15738897318/main,1,10,"1 file changed, 3 deletions(-)"
fork-NawazAli20/main,7,10,"1 file changed, 21 insertions(+), 17 deletions(-)"
fork-carlosrisma/main,2,1,"272 files changed, 1792 insertions(+), 20 deletions(-)"
fork-kaefcatcher/develop,6,0,"832 files changed, 1088402 insertions(+), 23 deletions(-)"
fork-kaefcatcher/main,24,0,"92 files changed, 12873479 insertions(+), 29 deletions(-)"
fork-maliksaad84/main,0,1,
fork-tfehren/main,7,10,"1 file changed, 21 insertions(+), 17 deletions(-)"
fork-vivatttt/aoi-aware-congestion-control,6,0,"56 files changed, 864 insertions(+), 279 deletions(-)"
fork-vivatttt/feature/aoi-aware-congestion-control-algorithm,1,0,"5 files changed, 242 insertions(+), 5 deletions(-)"


## kaefcatcher/MoReV2X_V2

Built for persistence (`Pkeep`) studies on SUMO traffic.

| Change | Where | Assessment |
|---|---|---|
| `Pkeep` attribute on the MAC | `nr-v2x-ue-mac.cc`, `--pkeep` | **Merge.** Upstream has `m_keepProbability` but no way to set it |
| Remove the live `std::cin.get()` in the keep branch | `nr-v2x-ue-mac.cc` | **Merge, required with `Pkeep`.** Otherwise any run with `Pkeep > 0` waits for a keypress |
| `run_simulator.py` and YAML configs | root, `simulations/` | **Port.** Useful parameter sweeps, but written in Python 2 and it spawns `./waf --run` in parallel, which conflicts over the waf lock |
| `results/metrics.py`, `kpi_for_pkeep.py` | `results/` | **Adopt the method.** PDR, PLR, CLR, PIR, AoI and PAoI per distance region. See [notebook 05](05_metrics.ipynb), which implements PDR, PIR and AoI |
| `return 1` in `GetCresel` | `nr-v2x-utils.cc:196` | **Do not merge.** Every grant lasts one period, so SPS is off for the whole fleet, and it makes `--pkeep` meaningless for runs with Cresel 1 |
| Commented out the assertion on latency reselections | `nr-v2x-ue-mac.cc` | **Do not merge.** It hides a broken invariant: periodic traffic should not need them |
| `T_gen = 100`, RRI forced to 100, 20 ms RRI dropped | `scratch/HIGHWAY.cc` | **Do not merge.** Hardcodes one traffic profile |
| CSV trace mobility: `opencsv`, `UpdateVehiclePositionsFromTrace` | `scratch/HIGHWAY.cc` | **Do not merge.** Positions are read by row order, so one vehicle more or less shifts every later vehicle onto the wrong node. Velocity is the scalar speed on x, so opposite directions look co-moving and `GetRelativeSpeed` is wrong. `--mobilityTrace` replaces it |
| `xml2csv/` SUMO scenarios, OSM map of Strogino | `xml2csv/` | **Take only if you want those maps.** The `.sumocfg` and `.osm` are reusable, the 12.8 million lines of generated traces are not worth carrying |
| `SSPS.ipynb`, `metrics_700_RC1.ipynb` | root | Reference only |

Note the interaction: with `GetCresel` returning 1 the counter expires after every transmission, so kaefcatcher's published `Pkeep` results describe a one-period scheme, not 3GPP SPS.

## vivatttt/MoReV2X

Branch `aoi-aware-congestion-control`. Adapts the RRI and the persistence probability from the free-subchannel ratio it senses, in a new class `AdaptiveResourceReservation`.

```cpp
// aoi-aware-congestion-control.cc: the core of UpdateRRI
if (pi0 < P0_INF) {            // 0.3: congested, slow down and persist more rarely
    R_tilde = std::min(R_tilde + DELTA_R, RRI_MAX);
    m_persistenceProbability = std::max(m_persistenceProbability - BETA, 0.0);
} else if (pi0 > P0_SUP) {     // 0.5: free channel, speed up
    R_tilde = std::max(R_tilde - DELTA_R, RRI_MIN);
    m_persistenceProbability = std::min(m_persistenceProbability + BETA, P_MAX);
}
R_tilde = BETA * R_tilde + (1.0 - BETA) * avgNeighborRRI;   // then pull towards the neighbours
```

| Change | Where | Assessment |
|---|---|---|
| `tmp1 + (10249 - tmp2)` becomes `10240` in `SubframeInfo::operator-` | `nist-sl-pool.h` | **Merge.** Real off-by-9 at the SFN wrap. Low impact today: no use of this operator was found in the module |
| Initialise members in the constructor | `nr-v2x-ue-mac.cc` | **Merge the idea, by hand.** `m_frameNo` and friends are read uninitialised. The patch also reformats the whole constructor |
| `AdaptiveResourceReservation` | new model files | **Rewritten**, see the merge list below |
| `m_RRIvalues` loop replaced by the grant's own RRI when excluding past transmissions | `Mode2Step1` | **Do not merge.** Step 1 must project own past transmissions over every configured RRI |
| `NrV2XUeMac::SubtractFrames` member | `nr-v2x-ue-mac.cc` | **Do not merge.** Shadows the free function and drops the wrap at SFN 1024, so the unsigned result underflows |
| New `SubtractFrames` in utils | `nr-v2x-utils.cc` | Equivalent to the original, no gain |
| `RandomSelection` attribute | `nr-v2x-ue-mac.cc` | **Do not merge.** Second name for `m_randomSelection`, and random selection still ends in `NS_FATAL_ERROR` |
| `std::cout` in `ComputeResidualCSRs` | `nr-v2x-utils.cc` | **Do not merge.** Prints during every selection |
| `AoIMetrics`, `CBRMetrics` | `scratch/HIGHWAY.cc` | **Adopt the method** in post-processing rather than online |
| `.pyc`, `waf`, `.lock-waf` | many | **Do not merge.** Build noise. `waf` was only regenerated with a `python2.7` shebang |

## carlosrisma/MoReV2X

A port of the module to ns-3.36, based on v4.1. Upstream has only changed its README since, so the code is current.

- `src/MoReV2X` renamed to `src/morev2x` (263 of 272 files are pure renames).
- `CMakeLists.txt` added, for the ns-3.36 build system.
- `HIGHWAY_2.cc` added to `examples/`.
- API fixes: `GetDevice() const`, `GetMobility() const`, `GetAntenna()`.
- **An unresolved TODO**: `SocketAddressTag` was removed in ns-3.26, so `nist-epc-sgw-pgw-application.cc` just comments out the removal of the tag. It affects the EPC path, not Mode 2 sidelink.

It only contains the module, on top of the old tree, so it cannot be merged here: it needs an ns-3.36 checkout and has not been built for this comparison.

Migration was assessed in [notebook 06](06_ns3_migration.ipynb), and the decision is to stay on ns-3.22. This port saved only 15 of 70 errors, so it would not be the starting point anyway.

## Merge list

| # | Change | Status | Branch |
|---|---|---|---|
| 1 | kaefcatcher: `Pkeep` attribute, removal of `std::cin.get()` | **Merged**, also per node through `pkeep=<p>` in `--SchedulerMap` | `feature/pkeep-attribute` |
| 2 | vivatttt: `operator-` fix | **Merged**, with a test that fails on the old code | `fix/sfn-wrap-subtraction` |
| 3 | vivatttt: constructor initialisation | **Merged** by hand, 6 members | `fix/mac-member-init` |
| 4 | `run_simulator.py` | **Rewritten** as `tools/run_sweep.py`: Python 3 in `.venv`, resumes, writes `summary.csv` | `feature/parameter-sweeps` |
| 5 | `AdaptiveResourceReservation` as an optional per-node scheduler | **Rewritten**, off by default, per node through `adaptive` in `--SchedulerMap` | `feature/adaptive-reservation` |

`tools/fingerprint.sh` gave the same hashes before and after items 1, 2, 3 and 5, with default settings. All went through `tools/gate.sh`.

### What changed in item 5

The algorithm is vivatttt's. The integration is new, because the original had these problems:

| Original | Now |
|---|---|
| The persistence probability was computed and never used | It drives `Pkeep` |
| The RRI changed only in the tag of aperiodic packets, so periodic traffic drifted from its grant | The packet period follows the RRI, so a change of RRI is also a change of rate |
| Neighbour RRIs included re-transmission offsets, stored in the same field | Those entries are skipped |
| Occupancy counted one RB range per entry over an RRI-long window, mixing 0-based and 1-based frames | Counts busy subchannel-slots over `AdaptiveWindow`, 100 ms |
| Any RRI value, list limit removed, Step 1 changed to the grant's RRI only | RRI on a 10 ms grid, added to the list, Step 1 unchanged |
| C++17, `std::cout` on every selection, hardcoded constants | C++11, `NS_LOG`, attributes |

Items marked **Do not merge** in the tables above stay out.

To see any change of a fork:
```bash
git show fork-kaefcatcher/main:src/MoReV2X/model/nr-v2x-utils.cc | sed -n 180,198p
git diff upstream/main fork-vivatttt/aoi-aware-congestion-control -- src/MoReV2X/model/nist-sl-pool.h
```
## Upstream changes to stock ns-3.22

For reference: against the official `ns-3.22` tag, upstream altered 13 stock sources, mainly `multi-model-spectrum-channel.cc` (adds a `Gain` trace), `propagation-loss-model.*`, `buildings-propagation-loss-model.*`, `callback.h` (supports more arguments), and `udp-client.*` (makes `Send` public, adds DENM counters).